# CELA.PRO — Sovereign AI OS v2026 (Colab)
الواجهة العربية (Gradio) مع قائمة اختيار المزوّد (`Groq` / `Gemini` / `OpenRouter` / `Kimi` / `Ollama`).

### إعداد الأسرار في Colab Secrets (🔑)
- `GROQ_API_KEY` — النموذج الافتراضي: `openai/gpt-oss-20b`
- `GEMINI_API_KEY` — النموذج الافتراضي: `gemini-2.5-flash`
- `OPENROUTER_API_KEY` — النموذج الافتراضي: `openrouter/free`
- `KIMI_API_KEY` — النموذج الافتراضي: `moonshot-v1-8k`
- `GRADIO_USERNAME` و `GRADIO_PASSWORD` — لحماية الرابط العام (`share=True`) من استهلاك حصص API.

In [ ]:
!pip install -q gradio httpx

In [ ]:
import os

SECRET_KEYS = (
    "GROQ_API_KEY",
    "GEMINI_API_KEY",
    "OPENROUTER_API_KEY",
    "KIMI_API_KEY",
    "GRADIO_USERNAME",
    "GRADIO_PASSWORD",
    "CELA_DEFAULT_PROVIDER",
    "GROQ_MODEL",
    "GEMINI_MODEL",
    "OPENROUTER_MODEL",
    "KIMI_MODEL",
    "OLLAMA_MODEL",
    "OLLAMA_BASE_URL",
)

try:
    from google.colab import userdata
except ImportError:
    userdata = None

loaded = []
for key in SECRET_KEYS:
    val = None
    if userdata is not None:
        try:
            val = userdata.get(key)
        except Exception:
            val = None
    if val:
        os.environ[key] = str(val).strip()
        loaded.append(key)

print("تم تحميل المتغيرات من Colab Secrets:", loaded or "(لم يُضبط أي سر بعد)")


In [ ]:
import pathlib
import subprocess

app_path = pathlib.Path("cela_pro_app.py")
if not app_path.exists():
    repo_url = "https://github.com/sayedelazameydesign-crypto/open_source-.git"
    subprocess.run(["git", "clone", repo_url, "cela_repo"], check=False)
    candidate = pathlib.Path("cela_repo/cela_pro_app.py")
    if candidate.exists():
        app_path.write_text(candidate.read_text(encoding="utf-8"), encoding="utf-8")

if not app_path.exists():
    try:
        from google.colab import files

        print("ارفع ملف cela_pro_app.py المرافق:")
        files.upload()
    except ImportError:
        pass

print("حالة ملف التطبيق:", "موجود ✅" if app_path.exists() else "غير موجود ❌")


In [ ]:
# خلية الفحص الدقيق (Smoke Test) — تُنفَّذ قبل تشغيل واجهة Gradio
# ترسل طلب تحقق واحدًا إلى Groq و Gemini وتتأكد من وصول ردٍّ حقيقي
# يتضمن عبارة «اختبار الاتصال ناجح».
# الحالات: BLOCKED (المفتاح غير مُعيّن) · FAIL (فشل الطلب أو رسالة خطأ)
#          INCONCLUSIVE (رد بلا عبارة التحقق) · PASS (رد يتضمن عبارة التحقق)
# قيمة مفتاح API تُحجب دائمًا [REDACTED] في المخرجات، ولا تبديل تلقائيًا بين المزوّدين.
import cela_pro_app

SMOKE_PROVIDERS = ("Groq", "Gemini")

smoke_summary = {}
for provider_name in SMOKE_PROVIDERS:
    result = cela_pro_app.run_smoke_test(provider_name)
    smoke_summary[provider_name] = result["state"]
    print(f"=== فحص {provider_name} — الحالة: {result['state']} ===")
    print(cela_pro_app.format_smoke_result(result))
    print()

print("ملخص الفحص الدقيق:", smoke_summary)
if all(state == "PASS" for state in smoke_summary.values()):
    print("✅ جميع المزوّدين المختبَرين جاهزون — تابع تشغيل الواجهة.")
else:
    print(
        "⚠️ يوجد مزوّد غير جاهز: راجع التفصيل أعلاه "
        "(مفتاح ناقص أو فشل اتصال أو رد بلا عبارة التحقق)."
    )


In [ ]:
import cela_pro_app

auth = cela_pro_app.get_launch_auth()
if auth is None:
    print(
        "⚠️ تحذير: أضف GRADIO_USERNAME و GRADIO_PASSWORD في Colab Secrets "
        "لحماية الرابط العام من استهلاك حصص API."
    )

cela_pro_app.demo.launch(share=True, auth=auth, css=cela_pro_app.CSS)
